# 复习包测试台

这个 notebook 用来测试现在的复习闭环：

- 知识点模式：先显示叶子卡片，再显示 2/3 道题，答案默认隐藏
- 题目模式：先显示题目，再显示关联知识点卡片
- 知识点反馈按钮：`掌握很熟练` / `还要练题`
- 题目反馈按钮：`做对` / `没做对` / `部分会做`

说明：
- 这里只往 `scratch/` 写测试文件
- 不会改正式题库
- 你可以直接替换输入的 `REVIEW_STATE_PATH` 来测试别的学生数据


In [ ]:
import copy
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from review_bundle_builder import (
    build_review_bundles,
    load_example_map,
    load_leaf_card_lookup,
)
from review_scheduler import load_review_state, now_from_value
from review_state_manager import apply_review_action

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'review_bundle_playground'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_STATE_PATH = PROJECT_ROOT / 'scratch' / 'student_annotation_merged_review_state.json'
EXAMPLES_MD_PATH = PROJECT_ROOT / 'docs' / 'rag_samples' / 'taizhou_simulated_exam_examples_batch_01.md'
DEMO_NOW = now_from_value('2026-06-21T10:00:00+08:00')

LEAF_BUNDLE_OUTPUT = SCRATCH_DIR / 'leaf_first_bundles.json'
QUESTION_BUNDLE_OUTPUT = SCRATCH_DIR / 'question_first_bundles.json'
NODE_MASTERED_OUTPUT = SCRATCH_DIR / 'after_node_mastered.json'
NODE_PRACTICE_OUTPUT = SCRATCH_DIR / 'after_node_needs_more_practice.json'


## 1. 载入当前复习状态

这里直接读取当前的 `review_state`。如果你想测新的学生数据，只要换掉上面的 `REVIEW_STATE_PATH`。

In [ ]:
REVIEW_STATE = load_review_state(REVIEW_STATE_PATH)
EXAMPLE_MAP = load_example_map(EXAMPLES_MD_PATH)
LEAF_CARD_LOOKUP = load_leaf_card_lookup()

print('record_id:', REVIEW_STATE.get('record_id'))
print('知识点数量:', len(REVIEW_STATE.get('knowledge_point_states', [])))
print('题目数量:', len(REVIEW_STATE.get('example_question_states', [])))

## 2. 生成知识点模式复习包

每个复习包包含：

- 一个叶子卡片
- 关联的 2/3 道题（如果暂时不够，就先展示现有题）
- 每道题的答案默认隐藏
- 可直接映射到 UI 的按钮动作


In [ ]:
LEAF_RESULT = build_review_bundles(
    REVIEW_STATE,
    example_map=EXAMPLE_MAP,
    leaf_card_lookup=LEAF_CARD_LOOKUP,
    now=DEMO_NOW,
    mode='leaf_first',
    bundle_limit=5,
    bundle_question_limit=3,
)

LEAF_BUNDLE_OUTPUT.write_text(
    json.dumps(LEAF_RESULT.as_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('知识点模式复习包已写入：', LEAF_BUNDLE_OUTPUT)
print()
for bundle in LEAF_RESULT.review_bundles:
    print(f"[{bundle['rank']}] 知识点：{bundle['leaf_card']['title'] if bundle['leaf_card'] else bundle['node_id']}")
    print('  原因：', bundle['bundle_reason'])
    print('  题目数量：', bundle['question_count'])
    if bundle['questions']:
        print('  第一道题：', (bundle['questions'][0]['content'].get('stem') or '')[:80])
        print('  答案默认隐藏：', bundle['questions'][0]['hidden_answer_block']['revealed'])
    print()

## 3. 生成题目模式复习包

这个模式适合“我先做题再看知识点”。

每个复习包包含：

- 一道题
- 相关知识点卡片
- 同样保留按钮动作结构


In [ ]:
QUESTION_RESULT = build_review_bundles(
    REVIEW_STATE,
    example_map=EXAMPLE_MAP,
    leaf_card_lookup=LEAF_CARD_LOOKUP,
    now=DEMO_NOW,
    mode='question_first',
    bundle_limit=5,
    bundle_question_limit=3,
)

QUESTION_BUNDLE_OUTPUT.write_text(
    json.dumps(QUESTION_RESULT.as_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('题目模式复习包已写入：', QUESTION_BUNDLE_OUTPUT)
print()
for bundle in QUESTION_RESULT.review_bundles:
    print(f"[{bundle['rank']}] 题目：{bundle['question']['question_id']}")
    print('  原因：', bundle['bundle_reason'])
    print('  题干预览：', (bundle['question']['content'].get('stem') or '')[:80])
    print('  关联知识点卡片数：', len(bundle['linked_leaf_cards']))
    print()

## 4. 测试知识点按钮反馈

这里模拟两个按钮：

- `掌握很熟练`：把这个知识点往后排
- `还要练题`：把这个知识点关联的题提上来


In [ ]:
TARGET_NODE_ID = 'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化'

mastered_event = apply_review_action(
    copy.deepcopy(REVIEW_STATE),
    action='node_mastered_well',
    target_type='knowledge_point',
    target_id=TARGET_NODE_ID,
    now=DEMO_NOW,
)
NODE_MASTERED_OUTPUT.write_text(
    json.dumps(mastered_event.as_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

practice_event = apply_review_action(
    copy.deepcopy(REVIEW_STATE),
    action='node_needs_more_practice',
    target_type='knowledge_point',
    target_id=TARGET_NODE_ID,
    now=DEMO_NOW,
)
NODE_PRACTICE_OUTPUT.write_text(
    json.dumps(practice_event.as_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('已写入：')
print('-', NODE_MASTERED_OUTPUT)
print('-', NODE_PRACTICE_OUTPUT)
print()

for node_state in mastered_event.updated_payload['knowledge_point_states']:
    if node_state['node_id'] == TARGET_NODE_ID:
        print('掌握很熟练后：')
        print(json.dumps(node_state, ensure_ascii=False, indent=2))
        break

print()

for node_state in practice_event.updated_payload['knowledge_point_states']:
    if node_state['node_id'] == TARGET_NODE_ID:
        print('还要练题后：')
        print(json.dumps(node_state, ensure_ascii=False, indent=2))
        break

## 5. 查看某个完整复习包的 JSON

这里直接看第一个知识点复习包，确认：

- 卡片内容是否够完整
- 题目答案是否确实是 hidden
- 按钮结构是否符合你想接 UI 的方式


In [ ]:
print(json.dumps(LEAF_RESULT.review_bundles[0], ensure_ascii=False, indent=2))